In [ ]:
!wget -qO- https://astral.sh/uv/install.sh | sh

!uv venv .venv --seed

!uv pip install -r requirements.txt

!uv pip install git+https://github.com/deepseek-ai/DeepGEMM.git --no-build-isolation

!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

In [1]:
import json
import os
import re
import sys
import random

from tqdm import tqdm
from pathlib import Path
from typing import Optional
import pandas as pd

import time
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

import copy
from collections import Counter, defaultdict

sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
DATA_PATH   = "data/private.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
ADAPTER_PATH = "./qwen3-4b-thinking-openr1-qlora-5k"
MAX_TOKENS = 32768

# os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

In [3]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

Loaded 943 questions  (300 MCQ, 643 free-form)


In [4]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem carefully. "
    "Do not use \\boxed{} except for the final answer. "
    "End with exactly one final line of the form Final answer: \\boxed{...}. "
    "For multiple [ANS] placeholders, put all answers in order inside one box, separated by commas. "
    "Prefer exact answers over decimal approximations whenever possible. "
    "If the answer naturally contains fractions, radicals, powers, logarithms, trigonometric values, or \\pi, "
    "keep the answer in exact symbolic form instead of converting it to a decimal. "
    "For numerical answers, do not round unless the problem explicitly says to round. "
    "If the problem says to use at least N decimal places, treat N as a minimum and output more precision, "
    "ideally 10-15 significant digits. "
    "Only round to exactly N decimal places if the problem explicitly says 'round to N decimal places', "
    "'nearest thousandth', 'nearest hundredth', or similar."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"

    n_ans = question.count("[ANS]")
    user_prompt = (
        f"This problem has exactly {n_ans} [ANS] placeholder(s). "
        f"Your final boxed answer must contain exactly {n_ans} answer(s), in the same order. "
        f"If there are multiple answers, separate them by commas inside one box.\n\n"
        "Important answer-format rules:\n"
        "- Prefer exact symbolic forms over decimals whenever possible.\n"
        "- Do not round unless the problem explicitly asks you to round.\n"
        "- If the problem asks for 'at least' some number of decimal places, give more precision than requested.\n"
        "- If decimals are necessary, use 10-15 significant digits when possible.\n"
        "- Do not include explanations, labels, or units inside the final box unless they are required by the answer.\n\n"
        f"{question}"
    )
    return SYSTEM_PROMPT_MATH, user_prompt

In [5]:
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.90,
    max_model_len=24576,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

print("Model loaded.")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.8k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=1859) /workspace/151B_SP26_Competition/.venv/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=1859)   torch._check_is_size(blocksize)


(EngineCore pid=1859) INFO 05-31 07:03:42 [gpu_model_runner.py:4879] Model loading took 2.7 GiB memory and 37.230555 seconds
(EngineCore pid=1859) INFO 05-31 07:04:16 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/ffedfa5cde/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=1859) INFO 05-31 07:04:16 [backends.py:1128] Dynamo bytecode transform time: 33.50 s
(EngineCore pid=1859) INFO 05-31 07:04:28 [backends.py:376] Cache the graph of compile range (1, 32768) for later use
(EngineCore pid=1859) INFO 05-31 07:04:37 [backends.py:391] Compiling a graph for compile range (1, 32768) takes 20.68 s
(EngineCore pid=1859) INFO 05-31 07:04:55 [decorators.py:668] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/0791840956d16125d8cf80c59217eef549099dea3701a5ca86b0d8ec7637fbc9/rank_0_0/model
(EngineCore pid=1859) INFO 05-31 07:04:55 [monitor.py:53] torch.compile took 72.28 s in total
(EngineCore pid=1859) INFO 05-31 07:

(EngineCore pid=1859) 2026-05-31 07:05:06,845 - INFO - autotuner.py:457 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=1859) 2026-05-31 07:05:06,936 - INFO - autotuner.py:466 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  98%|█████████▊| 50/51 [00:04<00:00, 11.38it/s]/workspace/151B_SP26_Competition/.venv/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=1859)   torch._check_is_size(blocksize)
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 11.43it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 13.88it/s]


(EngineCore pid=1859) INFO 05-31 07:05:14 [gpu_model_runner.py:6133] Graph capturing finished in 8 secs, took 0.70 GiB
(EngineCore pid=1859) INFO 05-31 07:05:14 [gpu_worker.py:599] CUDA graph pool memory: 0.7 GiB (actual), 0.49 GiB (estimated), difference: 0.2 GiB (28.9%).
(EngineCore pid=1859) INFO 05-31 07:05:14 [core.py:299] init engine (profile, create kv cache, warmup model) took 92.50 s (compilation: 72.28 s)
(EngineCore pid=1859) INFO 05-31 07:05:15 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
Model loaded.


In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_lora=True,
    max_lora_rank=64,
    enable_prefix_caching=False,
    gpu_memory_utilization=0.90,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

lora_request = LoRARequest(
    lora_name="numina_qlora",
    lora_int_id=1,
    lora_path=ADAPTER_PATH,
)

print("Model + adapter loaded.")

In [13]:
sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Sampling params loaded.")

Sampling params loaded.


In [6]:
def is_valid_response(item, response, judger):
    is_mcq = bool(item.get("options"))
    extracted = judger.extract_ans(response)

    if not extracted:
        return False

    if is_mcq:
        return bool(re.fullmatch(r"\s*[A-Z]\s*", extracted))

    expected = item["question"].count("[ANS]")
    pred_count = len(judger.split_by_comma(extracted))
    return pred_count == expected

In [7]:
# Generation strategy for majority voting
MCQ_NUM_SAMPLES = 1
FRQ_SINGLE_NUM_SAMPLES = 1
FRQ_MULTI_NUM_SAMPLES = 1

In [ ]:
def expected_answer_count(item):
    if is_mcq(item):
        return 1

    n = item["question"].count("[ANS]")

    # Some private FRQs appear to expect one answer even without [ANS]
    return max(1, n)


def is_mcq(item):
    return bool(item.get("options"))


def sample_count_for_item(item):
    """Choose number of generations based on question type."""
    if is_mcq(item):
        return MCQ_NUM_SAMPLES

    n_ans = expected_answer_count(item)
    if n_ans <= 1:
        return FRQ_SINGLE_NUM_SAMPLES

    return FRQ_MULTI_NUM_SAMPLES


def extract_answer_safe(response):
    try:
        return judger.extract_ans(response)
    except Exception:
        return ""


def normalize_frq_answer_key(extracted):
    """
    Convert extracted FRQ answer into a normalized tuple for voting.
    Returns None if normalization fails.
    """
    if not extracted:
        return None

    try:
        parts = judger.split_by_comma(extracted)
        return tuple(judger.norm_ans_str(p) for p in parts)
    except Exception:
        return None


def extract_mcq_letter_key(extracted):
    """
    Convert extracted MCQ answer into a single uppercase letter for voting.
    Returns None if invalid.
    """
    if not extracted:
        return None

    extracted = extracted.strip().upper()

    if re.fullmatch(r"[A-Z]", extracted):
        return extracted

    return None


def candidate_key(item, response):
    """
    Return a vote key for a candidate response.
    Invalid candidates return None.
    """
    extracted = extract_answer_safe(response)

    if is_mcq(item):
        return extract_mcq_letter_key(extracted)

    key = normalize_frq_answer_key(extracted)
    if key is None:
        return None

    if len(key) != expected_answer_count(item):
        return None

    return key


def is_valid_response(item, response):
    return candidate_key(item, response) is not None

In [9]:
# Build prompts
prompts = []

for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

print(f"Built {len(prompts)} prompts.")

Built 943 prompts.


In [16]:
def sampling_params_with_n(base_sampling_params, n):
    """
    Copy existing sampling params and set n completions per prompt.
    """
    params = copy.deepcopy(base_sampling_params)
    params.n = n
    return params


def generate_candidates_for_indices(indices, n, use_lora: bool):
    """
    Generate n candidate responses for each selected index.
    If use_lora=True, use the QLoRA adapter.
    If use_lora=False, use the base/non-QLoRA model.
    Returns dict: index -> list[str].
    """
    if not indices:
        return {}

    selected_prompts = [prompts[i] for i in indices]
    params = sampling_params_with_n(sampling_params, n)

    # if use_lora:
    #     outputs = llm.generate(
    #         selected_prompts,
    #         sampling_params=params,
    #         lora_request=lora_request,
    #     )
    # else:
    #     outputs = llm.generate(
    #         selected_prompts,
    #         sampling_params=params,
    #     )
    outputs = llm.generate(selected_prompts, sampling_params=params)

    results = {}
    for idx, out in zip(indices, outputs):
        results[idx] = [candidate.text.strip() for candidate in out.outputs]

    return results


def generate_candidates_grouped(indices):
    """
    Group by:
    1. MCQ vs FRQ, because MCQ should use base model and FRQ should use QLoRA.
    2. sample count, because vLLM requires one n value per generate call.

    Returns dict: index -> list[str].
    """
    grouped = defaultdict(list)

    for idx in indices:
        item = data[idx]
        n = sample_count_for_item(item)

        # MCQ -> base model, FRQ -> QLoRA
        use_lora = not is_mcq(item)

        grouped[(use_lora, n)].append(idx)

    all_results = {}

    for (use_lora, n), group_indices in sorted(grouped.items()):
        model_name = "QLoRA" if use_lora else "base"
        print(
            f"Generating {n} sample(s) each for "
            f"{len(group_indices)} questions using {model_name} model..."
        )

        group_results = generate_candidates_for_indices(
            group_indices,
            n,
            use_lora=use_lora,
        )

        all_results.update(group_results)

    return all_results

In [11]:
def choose_best_candidate(item, candidates):
    """
    Pick the best candidate using validity filtering + majority vote.

    Returns:
        chosen_response: str
        info: dict with vote metadata
    """
    valid = []

    for pos, response in enumerate(candidates):
        key = candidate_key(item, response)
        if key is not None:
            valid.append({
                "pos": pos,
                "response": response,
                "key": key,
            })

    # If no valid candidates, fall back to the first raw candidate.
    if not valid:
        return candidates[0] if candidates else "", {
            "valid_count": 0,
            "num_candidates": len(candidates),
            "winning_key": None,
            "vote_count": 0,
        }

    # Count votes by normalized answer key.
    counts = Counter(v["key"] for v in valid)
    winning_key, vote_count = counts.most_common(1)[0]

    # Use first valid candidate that produced the winning key.
    for v in valid:
        if v["key"] == winning_key:
            return v["response"], {
                "valid_count": len(valid),
                "num_candidates": len(candidates),
                "winning_key": winning_key,
                "vote_count": vote_count,
            }

    # Should never reach here.
    return valid[0]["response"], {
        "valid_count": len(valid),
        "num_candidates": len(candidates),
        "winning_key": valid[0]["key"],
        "vote_count": 1,
    }


def select_responses_from_candidates(candidate_map):
    """
    Convert index -> candidates into final selected responses.
    Returns:
        selected: dict index -> response
        vote_info: dict index -> metadata
    """
    selected = {}
    vote_info = {}

    for idx, candidates in candidate_map.items():
        response, info = choose_best_candidate(data[idx], candidates)
        selected[idx] = response
        vote_info[idx] = info

    return selected, vote_info

In [17]:
print(f"Generating initial candidates for {len(prompts)} questions...")

responses = [""] * len(prompts)
all_vote_info = {}

initial_indices = list(range(len(prompts)))
initial_candidate_map = generate_candidates_grouped(initial_indices)

initial_selected, initial_vote_info = select_responses_from_candidates(initial_candidate_map)

for idx, response in initial_selected.items():
    responses[idx] = response
    all_vote_info[idx] = initial_vote_info[idx]

initial_valid = sum(is_valid_response(data[i], responses[i]) for i in range(len(responses)))
print(f"Initial valid selected responses: {initial_valid}/{len(responses)}")

Generating initial candidates for 943 questions...
Generating 1 sample(s) each for 300 questions using base model...


Rendering prompts:   0%|          | 0/300 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/300 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating 1 sample(s) each for 643 questions using QLoRA model...


Rendering prompts:   0%|          | 0/643 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/643 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Initial valid selected responses: 887/943


In [15]:
valid_count = 0
invalid_count = 0

vote_valid_counts = Counter()
vote_agreement_counts = Counter()

for i, response in enumerate(responses):
    if is_valid_response(data[i], response):
        valid_count += 1
    else:
        invalid_count += 1

    info = all_vote_info.get(i, {})
    vote_valid_counts[info.get("valid_count", 0)] += 1
    vote_agreement_counts[info.get("vote_count", 0)] += 1

print(f"Final valid responses: {valid_count}/{len(responses)}")
print(f"Final invalid responses: {invalid_count}/{len(responses)}")

print("\nValid candidate counts among selected generations:")
for k, v in sorted(vote_valid_counts.items()):
    print(f"  {k} valid candidate(s): {v}")

print("\nWinning vote counts:")
for k, v in sorted(vote_agreement_counts.items()):
    print(f"  winning vote count {k}: {v}")

Final valid responses: 0/943
Final invalid responses: 943/943

Valid candidate counts among selected generations:
  0 valid candidate(s): 943

Winning vote counts:
  winning vote count 0: 943


In [ ]:
def normalize_choice_text(s: str) -> str:
    """
    Normalize option/model text for matching:
    - remove surrounding $$
    - remove whitespace
    - remove common final punctuation
    - lowercase
    """
    if s is None:
        return ""

    s = str(s).strip()

    # Remove common wrappers
    s = s.strip()
    s = s.strip("$")
    s = s.strip()

    # Remove leading choice labels if present, e.g. "A. ...", "B: ..."
    s = re.sub(r"^\s*[A-Z]\s*[\.\:\)]\s*", "", s, flags=re.IGNORECASE)

    # Normalize whitespace and punctuation
    s = re.sub(r"\s+", "", s)
    s = s.rstrip(".,;:")
    s = s.lower()

    return s


def extract_letter_from_mcq_answer(extracted: str, options: list[str]):
    """
    Return answer letter if extracted is already a letter or exactly matches
    one of the option texts. Otherwise return None.
    """
    if not extracted:
        return None

    s = extracted.strip()

    # Already a letter: C
    if re.fullmatch(r"[A-Z]", s, flags=re.IGNORECASE):
        return s.upper()

    # Bracketed letter: [C]
    m = re.fullmatch(r"\[\s*([A-Z])\s*\]", s, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()

    # Boxed-looking content may have C: or C.
    m = re.fullmatch(r"([A-Z])\s*[\.\:\)]", s, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()

    # Sometimes extraction gives "C. <choice text>"
    m = re.match(r"^\s*([A-Z])\s*[\.\:\)]\s*(.+)$", s, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()

    # Match full extracted answer to option text
    norm_extracted = normalize_choice_text(s)

    for i, opt in enumerate(options):
        if norm_extracted == normalize_choice_text(opt):
            return chr(65 + i)

    return None


def replace_final_answer_with_letter(response: str, letter: str) -> str:
    """
    Replace the final answer in the response with \\boxed{LETTER}.
    Prefer replacing the last \\boxed{...}; otherwise append a clean final line.
    """
    if not response:
        return f"Final answer: \\boxed{{{letter}}}"

    # Replace the last \boxed{...}, supporting nested braces lightly
    boxed_positions = [m.start() for m in re.finditer(r"\\boxed\{", response)]

    if boxed_positions:
        start = boxed_positions[-1]
        content_start = start + len(r"\boxed{")

        depth = 1
        i = content_start
        while i < len(response) and depth > 0:
            if response[i] == "{":
                depth += 1
            elif response[i] == "}":
                depth -= 1
            i += 1

        if depth == 0:
            return response[:start] + f"\\boxed{{{letter}}}" + response[i:]

    # If no valid boxed answer exists, append one
    return response.rstrip() + f"\nFinal answer: \\boxed{{{letter}}}"


postprocessed_responses = list(responses)
num_mcq_seen = 0
num_mcq_changed = 0
num_mcq_unmatched = 0

for i, item in enumerate(data):
    if not item.get("options"):
        continue

    num_mcq_seen += 1
    response = postprocessed_responses[i]

    try:
        extracted = judger.extract_ans(response)
    except Exception:
        extracted = ""

    letter = extract_letter_from_mcq_answer(extracted, item["options"])

    if letter is None:
        num_mcq_unmatched += 1
        continue

    # Only rewrite if extracted was not already exactly the clean letter
    if extracted.strip().upper() != letter:
        postprocessed_responses[i] = replace_final_answer_with_letter(response, letter)
        num_mcq_changed += 1

print(f"MCQ seen: {num_mcq_seen}")
print(f"MCQ postprocessed/replaced: {num_mcq_changed}")
print(f"MCQ unmatched: {num_mcq_unmatched}")

# Optional: use this as the final responses object for saving
responses = postprocessed_responses

In [86]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 197/197 [00:30<00:00,  6.44it/s]

Scoring complete. 197 results.


In [87]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    0 /    0  (0.00%)
  Free-form  :  116 /  197  (58.88%)
  Overall    :  116 /  197  (58.88%)


In [102]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 197 records to results/starter_results.jsonl


In [101]:
import pandas as pd
SUBMISSION_PATH = "submission2.csv"

# Build submission rows from generated responses.
# Assumes:
# - data is the loaded private.jsonl/public.jsonl list of dicts
# - responses[i] is the selected full model response for data[i]
submission_df = pd.DataFrame({
    "id": [item["id"] for item in data],
    "response": responses,
})

# Basic checks
assert len(submission_df) == len(data), "Submission row count does not match data length."
assert submission_df["id"].isna().sum() == 0, "Some ids are missing."
assert submission_df["response"].isna().sum() == 0, "Some responses are missing."

# Optional: check for empty responses
empty_count = (submission_df["response"].astype(str).str.len() == 0).sum()
print(f"Empty responses: {empty_count}")

# Write CSV with proper quoting/escaping handled by pandas
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"Wrote {len(submission_df)} rows to {SUBMISSION_PATH}")
submission_df.head()

Empty responses: 0
Wrote 943 rows to submission2.csv


,id,response
0,0,"Okay, let's tackle these two problems step by ..."
1,1,"Okay, let's try to figure out this problem. Hm..."
2,2,"Okay, let me try to work through this problem ..."
3,3,"Okay, let's try to figure out this problem ste..."
4,4,"Okay, let's see. The problem says that the poi..."


In [103]:
JSONL_OUTPUT_PATH = "submission.jsonl"

# Assumes:
# - data is the loaded private.jsonl/public.jsonl list of dicts
# - responses[i] is the selected full model response for data[i]

rows = []
for item, response in zip(data, responses):
    rows.append({
        "id": item["id"],
        "response": response,
    })

# Basic checks
assert len(rows) == len(data), "JSONL row count does not match data length."
assert all("id" in row for row in rows), "Some rows are missing id."
assert all("response" in row for row in rows), "Some rows are missing response."
assert all(row["response"] is not None for row in rows), "Some responses are None."

empty_count = sum(len(str(row["response"])) == 0 for row in rows)
print(f"Empty responses: {empty_count}")

# Write JSONL: one JSON object per line
with open(JSONL_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Wrote {len(rows)} rows to {JSONL_OUTPUT_PATH}")

# Preview first few rows
for row in rows[:3]:
    print(row["id"], str(row["response"])[:200].replace("\n", "\\n"))

Empty responses: 0
Wrote 943 rows to submission.jsonl
0 Okay, let's tackle these two problems step by step. Starting with part a: [13 - (11 - 11)] - [8 - (5 - 6)]. Hmm, parentheses can be tricky, so I need to handle them carefully. Let me break it down.\n\nF
1 Okay, let's try to figure out this problem. Hmm, the question says: "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ()". Wait, the problem state
2 Okay, let me try to work through this problem step by step. So, the question is about hypothesis testing for a proportion. We have a sample of 120 drinkers, and 52 preferred Diet Pepsi. The anouncer s
